# Notebook 3: Exploring Protein Tertiary Structure

## Bio 4525: Structural Bioinformatics of Proteins
**Washington University in St. Louis**

---

## From Secondary to Tertiary Structure

In Notebook 2, we analyzed **secondary structure** — the local repeating patterns (helices, sheets, loops) that emerge from hydrogen bonds between backbone atoms. **Tertiary structure** is the next level: the complete three-dimensional arrangement of *all* atoms in a single polypeptide chain.

How does a polypeptide fold into a specific 3D shape? The driving forces are the **stabilizing interactions** that form between amino acid side chains once the protein collapses:

| Interaction | Description | Strength |
|---|---|---|
| **Hydrophobic effect** | Nonpolar side chains cluster in the protein interior, away from water. Each contact is weak, but dozens of buried residues together can **rival or exceed the strength of a covalent bond** — making this the dominant folding force. | Very strong (collective) |
| **Disulfide bonds** | Covalent S–S bonds between Cysteine side chains. Found mainly in secreted/extracellular proteins. | Strong (covalent) |
| **Ionic interactions** | Electrostatic attraction between oppositely charged side chains (Arg/Lys ↔ Asp/Glu). Also widely known as **salt bridges** in the structural biology literature. | Moderate |
| **Hydrogen bonds** | Between polar side chains, or between a side chain and the backbone. | Moderate |
| **Van der Waals** | Weak, short-range attractions. Large cumulative effect when many atoms are packed tightly. | Weak (but numerous) |

In this notebook we use the **RNase A crystal structure** (PDB: **8RAT**) to detect and visualize these interactions computationally. RNase A is an ideal choice: it is a secreted enzyme with **four disulfide bonds**, multiple ionic interactions, and a single well-defined structural domain.

## Learning Objectives

By the end of this notebook, you will be able to:

- Access the x, y, z coordinates of individual atoms from a PDB structure
- Calculate the Euclidean distance between two atoms using the 3D distance formula
- Write a reusable `calculate_distance()` function with a proper docstring
- Identify disulfide bonds by detecting close Cys SG atom pairs
- Identify ionic interactions between oppositely charged residue side chains
- Understand how protein domains are defined and classified (CATH/SCOP databases)
- Use nglview to display structures with different representations, color schemes, and atom selections

**Estimated time:** 60–75 minutes

In [ ]:
# -----------------------------------------------------------------------
# SETUP: Install and import all required libraries
# Run this cell first — it only needs to run once per Colab session
#
# IMPORTANT: After this cell finishes, go to:
#   Runtime → Restart session
# Then run ALL cells again from the top.
# -----------------------------------------------------------------------

!pip install biopython nglview==3.0.8 --quiet

from google.colab import output
output.enable_custom_widget_manager()

import os                          # file and path operations
import shutil                      # check if programs are installed
import requests                    # HTTP requests for web API calls
import numpy as np                 # numeric arrays and math
import pandas as pd                # data tables
import matplotlib.pyplot as plt    # charts and graphs
import nglview as nv               # interactive 3D molecular viewer

from Bio.PDB import PDBList, PDBParser    # download and parse PDB files

print('All libraries imported successfully!')
print(f'nglview version: {nv.__version__}')

---
## Section 1: Working with 3D Coordinates

### From Sequence to Space

A PDB file stores the **x, y, z coordinates** of every non-hydrogen atom in the crystal structure, measured in **Ångströms** (Å; 1 Å = 10⁻¹⁰ m). These coordinates are the raw material of structural bioinformatics: from them, we can calculate distances, angles, and volumes that reveal how the protein is held together.

### The Distance Formula in 3D

The straight-line distance between two points in 3D space is an extension of the Pythagorean theorem:

$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2 + (z_2 - z_1)^2}$$

This is called the **Euclidean distance**. In structural biology, distances are always in Ångströms:

- A covalent bond length: ~1.5 Å (C–C), ~1.0 Å (C–H)
- A disulfide bond (S–S): ~2.0 Å
- A hydrogen bond donor–acceptor: ~2.8–3.5 Å
- An ionic interaction (charged groups): ~2.5–4.0 Å
- Alpha carbon to alpha carbon of adjacent residues: ~3.8 Å

Knowing typical distances allows us to identify specific interactions by scanning all atom pairs and checking whether their distance falls in the expected range.

In [ ]:
# -----------------------------------------------------------------------
# Download and parse the RNase A structure (8RAT) from the PDB
# We use file_format='pdb' (legacy format) rather than 'mmCif' because
# the DSSP program installed via apt-get in Colab has unreliable mmCIF
# support. For small proteins like RNase A this is fine — the PDB
# format size limit (99,999 atoms) is not a concern here.
# -----------------------------------------------------------------------

pdbl = PDBList()

# The try/except block catches network errors and gives a clear message
try:
    downloaded_path = pdbl.retrieve_pdb_file(
        '8RAT',
        file_format='pdb',
        pdir='.'
    )
except Exception as download_error:
    raise RuntimeError(
        'Could not download 8RAT from RCSB PDB.\n'
        'Check your internet connection and try again.\n'
        f'Original error: {download_error}'
    )

print(f'Downloaded file path: {downloaded_path}')

# Search for the file if it is not at the expected path
if not os.path.exists(downloaded_path):
    print('File not at expected path, searching...')
    for root, dirs, files in os.walk('.'):
        for fname in files:
            if '8rat' in fname.lower():
                downloaded_path = os.path.join(root, fname)
                print(f'Found at: {downloaded_path}')
                break

pdb_filename = downloaded_path

# Parse the PDB file into a BioPython structure object
parser = PDBParser(QUIET=True)
structure = parser.get_structure('8RAT', pdb_filename)

# Select Model 0 (the only model in this X-ray structure)
model = structure[0]

print()
print('Structure parsed successfully!')
print(f'Chains in 8RAT: {[chain.id for chain in model.get_chains()]}')


---
## nglview Quick Reference

Before diving into the structural analysis, here is a short guide to the three most important concepts in nglview: **representations**, **color schemes**, and **selections**. Bookmark this section — you will use these tools in every visualization throughout this notebook series.

### 1. Representations

A representation controls how atoms are drawn. Specify it with the `representations=` parameter at load time, or add one on top of an existing view with `view.add_*()`:

| Representation | Python call | Best used for |
|---|---|---|
| `cartoon` | `view.add_cartoon()` | Overall fold — helices as ribbons, strands as arrows, loops as tubes |
| `ball_and_stick` | `view.add_ball_and_stick()` | Individual side chains, ligands, and active sites |
| `licorice` | `view.add_licorice()` | Side chains without atom spheres (cleaner look) |
| `surface` | `view.add_surface()` | Molecular surface — shows protein shape |
| `spacefill` | `view.add_spacefill()` | Each atom drawn at its full van der Waals radius |

### 2. Color Schemes

Pass `colorScheme` (not `color`) to apply a named coloring rule:

| `colorScheme` value | Colors by |
|---|---|
| `'sstruc'` | DSSP secondary structure (red=helix, yellow=strand, green=turn, white=coil) |
| `'chainindex'` | Chain identity — one distinct color per chain |
| `'residueindex'` | Rainbow from N-terminus (blue) → C-terminus (red) |
| `'element'` | Element type: C=gray, N=blue, O=red, S=yellow |
| `'bfactor'` | Crystallographic B-factor (a measure of atomic mobility) |

To use a single solid color instead of a scheme, pass `color='red'` or `color='#4472C4'`.

### 3. Selections

All `add_*` methods accept an optional selection string as the first argument. This limits the representation to a subset of atoms using NGL's selection language:

| Selection string | Selects |
|---|---|
| `':A'` | All atoms in chain A |
| `':A or :B'` | Chains A and B combined |
| `'[HIS]'` | All histidine residues (3-letter code in square brackets) |
| `'[HIS] or [LYS]'` | All His and Lys residues |
| `'12:A'` | Residue 12 in chain A |
| `'12-20:A'` | Residues 12 through 20 in chain A |
| `'.CA'` | Only the alpha carbon (CA) atoms |
| `'[CYS] and .SG'` | Only the SG (sulfur gamma) atoms of Cys residues |

> **Tip:** Combine these to build targeted views. For example, `view.add_ball_and_stick('[HIS]', colorScheme='element')` shows all histidine residues as ball-and-stick with CPK element colors layered on top of whatever backbone representation is already shown.

In [ ]:
# -----------------------------------------------------------------------
# nglview tutorial: four examples demonstrating representations,
# color schemes, and selections on the RNase A structure.
#
# Each example calls display() so all four viewers appear in one cell.
# Run this cell after downloading 8RAT in the cell above.
# -----------------------------------------------------------------------

from IPython.display import display   # allows showing multiple widgets

# -----------------------------------------------------------------------
# EXAMPLE 1: Cartoon colored by secondary structure (sstruc)
# This is the most common representation for understanding protein folds.
# -----------------------------------------------------------------------
print('Example 1: Cartoon — colored by secondary structure (sstruc)')
view1 = nv.show_structure_file(
    pdb_filename,
    representations=[{'type': 'cartoon', 'params': {'colorScheme': 'sstruc'}}]
)
view1.background = 'white'
display(view1)

# -----------------------------------------------------------------------
# EXAMPLE 2: Surface representation
# Shows the shape of the protein as the solvent would "see" it.
# -----------------------------------------------------------------------
print()
print('Example 2: Surface — colored by chain index')
view2 = nv.show_structure_file(
    pdb_filename,
    representations=[{'type': 'surface', 'params': {'colorScheme': 'chainindex', 'opacity': 0.8}}]
)
view2.background = 'white'
display(view2)

# -----------------------------------------------------------------------
# EXAMPLE 3: N→C rainbow coloring (residueindex)
# The default NGL coloring. Compare to sstruc above — same structure,
# completely different color meaning. This is why it matters which
# colorScheme you choose.
# -----------------------------------------------------------------------
print()
print('Example 3: Cartoon — N→C rainbow (residueindex), the default NGL coloring')
view3 = nv.show_structure_file(
    pdb_filename,
    representations=[{'type': 'cartoon', 'params': {'colorScheme': 'residueindex'}}]
)
view3.background = 'white'
display(view3)

# -----------------------------------------------------------------------
# EXAMPLE 4: Selections
# Show the backbone lightly, then highlight the three catalytic residues
# of RNase A (His12, Lys41, His119) as ball-and-stick with element colors.
# Selection syntax: '12:A' = residue 12 on chain A
# -----------------------------------------------------------------------
print()
print('Example 4: Selections — backbone cartoon + catalytic residues (His12, Lys41, His119)')
view4 = nv.show_structure_file(
    pdb_filename,
    representations=[{'type': 'cartoon', 'params': {'colorScheme': 'sstruc', 'opacity': 0.35}}]
)
# Add ball-and-stick only for the three catalytic residues
view4.add_ball_and_stick('12:A or 41:A or 119:A', colorScheme='element')
view4.background = 'white'
display(view4)

In [ ]:
# -----------------------------------------------------------------------
# Accessing 3D coordinates of individual atoms
#
# BioPython's SMCRA hierarchy lets us navigate down to a single atom:
#   structure[model_number][chain_id][residue_id][atom_name]
#
# Each atom has a .coord attribute that returns a NumPy array [x, y, z]
# in Angstroms.
# -----------------------------------------------------------------------

# RNase A (8RAT) has a single chain: Chain A with 124 residues.
chain_a = model['A']

# --- Example 1: the alpha carbon (CA) of Lysine 1 (N-terminus) ---
# Residue IDs are tuples: (' ', seq_number, ' ')
# The space in position 0 means standard amino acid (not a ligand/water)
residue_1 = chain_a[(' ', 1, ' ')]
atom_ca_res1 = residue_1['CA']
coord_ca_res1 = atom_ca_res1.coord    # NumPy array: [x, y, z] in Angstroms

print('Residue 1 of Chain A (Lys1 — N-terminus):')
print(f'  Residue name     : {residue_1.get_resname()}')
print(f'  Alpha carbon (CA): x={coord_ca_res1[0]:.3f}, y={coord_ca_res1[1]:.3f}, z={coord_ca_res1[2]:.3f} Å')
print()

# --- Example 2: the alpha carbon of residue 124 (C-terminus) ---
residue_124 = chain_a[(' ', 124, ' ')]
atom_ca_res124 = residue_124['CA']
coord_ca_res124 = atom_ca_res124.coord

print(f'Residue 124 of Chain A ({residue_124.get_resname()}124 — C-terminus):')
print(f'  Alpha carbon (CA): x={coord_ca_res124[0]:.3f}, y={coord_ca_res124[1]:.3f}, z={coord_ca_res124[2]:.3f} Å')
print()

# --- Example 3: His12 — the first catalytic histidine of the active site ---
# His12 and His119 act as general acid/base catalysts to cleave RNA.
residue_his12 = chain_a[(' ', 12, ' ')]
atom_ca_his12 = residue_his12['CA']
coord_his12 = atom_ca_his12.coord

print('Residue 12 of Chain A (His12 — catalytic histidine):')
print(f'  Residue name     : {residue_his12.get_resname()}')
print(f'  Alpha carbon (CA): x={coord_his12[0]:.3f}, y={coord_his12[1]:.3f}, z={coord_his12[2]:.3f} Å')

In [ ]:
# -----------------------------------------------------------------------
# Applying the 3D distance formula step by step
#
# Let's compute the distance between Lys1-CA and the C-terminal residue
# CA manually first, so you can see every arithmetic step.
# -----------------------------------------------------------------------

print(f'Step-by-step distance calculation: Lys1 CA  →  {residue_124.get_resname()}124 CA')
print('=' * 60)

# Step 1: extract x, y, z for each atom
x1, y1, z1 = coord_ca_res1
x2, y2, z2 = coord_ca_res124

print(f'  Atom 1 (Lys1 CA)   : ({x1:.3f}, {y1:.3f}, {z1:.3f})')
print(f'  Atom 2 (res124 CA)  : ({x2:.3f}, {y2:.3f}, {z2:.3f})')
print()

# Step 2: compute the difference in each coordinate
delta_x = x2 - x1
delta_y = y2 - y1
delta_z = z2 - z1

print(f'  Δx = {x2:.3f} - {x1:.3f} = {delta_x:.3f}')
print(f'  Δy = {y2:.3f} - {y1:.3f} = {delta_y:.3f}')
print(f'  Δz = {z2:.3f} - {z1:.3f} = {delta_z:.3f}')
print()

# Step 3: square each difference
delta_x_sq = delta_x ** 2
delta_y_sq = delta_y ** 2
delta_z_sq = delta_z ** 2

print(f'  Δx² = {delta_x_sq:.3f}')
print(f'  Δy² = {delta_y_sq:.3f}')
print(f'  Δz² = {delta_z_sq:.3f}')
print()

# Step 4: sum the squares and take the square root
sum_of_squares = delta_x_sq + delta_y_sq + delta_z_sq
distance_manual = np.sqrt(sum_of_squares)

print(f'  √(Δx² + Δy² + Δz²) = √({sum_of_squares:.3f}) = {distance_manual:.3f} Å')
print()
print(f'  Distance from N-terminus (Lys1) to C-terminus (res124): {distance_manual:.2f} Å')

In [ ]:
# -----------------------------------------------------------------------
# Write a reusable calculate_distance() function
# -----------------------------------------------------------------------

def calculate_distance(atom1, atom2):
    '''
    Calculate the Euclidean distance between two atoms in 3D space.

    Uses the formula: d = sqrt((x2-x1)^2 + (y2-y1)^2 + (z2-z1)^2)

    Parameters:
    -----------
    atom1 : Bio.PDB.Atom.Atom
        The first atom (must have a .coord attribute)
    atom2 : Bio.PDB.Atom.Atom
        The second atom (must have a .coord attribute)

    Returns:
    --------
    float
        Distance in Angstroms (Å)
    '''
    coord1 = atom1.coord
    coord2 = atom2.coord
    diff_vector = coord2 - coord1
    distance = np.sqrt(np.sum(diff_vector ** 2))
    return distance


print('calculate_distance() function defined.')
print()

# --- Verify against the manual calculation ---
dist_n_to_c = calculate_distance(atom_ca_res1, atom_ca_res124)
print(f'N-terminus (Lys1) → C-terminus (res124): {dist_n_to_c:.2f} Å')
print()

# --- Active site: His12 CA → His119 CA ---
# His12 and His119 are the two histidines that catalyze RNA cleavage.
# Both must reach the same RNA substrate, so their CA–CA distance
# gives us a rough sense of the active site dimensions.
residue_his119 = chain_a[(' ', 119, ' ')]
atom_ca_his119 = residue_his119['CA']

dist_his12_his119 = calculate_distance(atom_ca_his12, atom_ca_his119)
print(f'His12 CA → His119 CA (active site histidines): {dist_his12_his119:.2f} Å')
print()

# --- Active site: Lys41 ---
# Lys41 is the third catalytic residue; it stabilizes the transition state.
residue_lys41 = chain_a[(' ', 41, ' ')]
atom_ca_lys41 = residue_lys41['CA']

dist_his12_lys41 = calculate_distance(atom_ca_his12, atom_ca_lys41)
print(f'His12 CA → Lys41 CA (His12 to the catalytic lysine): {dist_his12_lys41:.2f} Å')
print()

# --- Sanity check: adjacent CA atoms should be ~3.8 Å ---
residue_2 = chain_a[(' ', 2, ' ')]
atom_ca_res2 = residue_2['CA']
dist_1_to_2 = calculate_distance(atom_ca_res1, atom_ca_res2)
print(f'Lys1 CA → {residue_2.get_resname()}2 CA (adjacent residues, should be ~3.8 Å): {dist_1_to_2:.2f} Å')

### 💭 Think About It

1. **The distance from Lys1-CA to res124-CA is the straight-line distance** between the N and C termini of RNase A. If the 124-residue chain were fully extended (adjacent CA atoms 3.8 Å apart), what would the total end-to-end distance be? How does the actual measured distance compare? What does the difference tell you about how tightly the chain is folded?

2. **His12 and His119 are both required for catalysis** — they must both contact the same RNA substrate. Based on the CA–CA distance you measured, are they close enough to cooperate in the active site? (The catalytic atoms are the side-chain NE2 nitrogens, which are typically 3–5 Å closer to each other than the CA atoms.)

3. **Adjacent CA atoms are ~3.8 Å apart** regardless of the protein. This is a direct consequence of the peptide bond geometry. Why would this invariance be useful as a quality check when reading PDB coordinates?

4. **Lys41 is the third catalytic residue** of RNase A. Based on the distance you measured between His12-CA and Lys41-CA, does Lys41 appear to be in the same general region of the protein as the two histidines? Can you verify this by looking at the nglview visualization from the tutorial above?

---
## Section 2: Finding Disulfide Bonds

### What is a Disulfide Bond?

A **disulfide bond** is a covalent bond between the sulfur atoms of two Cysteine residues. The reaction is an oxidation:

&nbsp;&nbsp;&nbsp;&nbsp;2 Cys–SH → Cys–S–S–Cys + 2H⁺ + 2e⁻

Key features:
- The theoretical S–S bond length is approximately **2.03–2.05 Å**, but crystal structures at typical X-ray resolution (1.5–3.0 Å) can report values up to **~2.15 Å** due to coordinate uncertainty. We use a threshold of **2.2 Å** to reliably detect disulfide bonds while still excluding non-bonded Cys pairs (which are usually >4 Å apart).
- Disulfide bonds form readily in **oxidizing environments** (extracellular space, lysosomes, endoplasmic reticulum)
- They are rare in **intracellular proteins** (the cytoplasm is reducing — free thiols are maintained by thioredoxin and glutathione)

### RNase A: A Classic Disulfide-Bonded Protein

RNase A is a **secreted enzyme** — it is made in the pancreas and exported into the small intestine. As a protein destined for an oxidizing environment, it has **four disulfide bonds** involving all 8 of its Cysteine residues:

| Bond | Residues | Notes |
|------|----------|-------|
| 1 | Cys26–Cys84 | Cross-links a helix to the beta sheet core |
| 2 | Cys40–Cys95 | Stabilizes a loop region |
| 3 | Cys58–Cys110 | Cross-links two beta strands |
| 4 | Cys65–Cys72 | Forms a small disulfide loop within a strand |

These four bonds are the same ones Christian Anfinsen used in his Nobel Prize-winning experiment: he denatured RNase A, scrambled all the disulfide bonds, then showed that the correct four bonds reformed spontaneously when the protein refolded — proving that sequence alone encodes structure.

We will write a function to detect all four bonds computationally.

In [ ]:
# -----------------------------------------------------------------------
# Write a function to detect potential disulfide bonds
#
# Strategy:
#   1. Find every Cysteine residue in the structure
#   2. Get its SG (sulfur gamma) atom
#   3. For EVERY unique pair of SG atoms, calculate and PRINT the distance
#   4. Flag pairs where the distance is <= threshold as disulfide bonds
#
# Why 2.2 Å and not 2.05 Å (the theoretical S–S bond length)?
# X-ray crystal structures have coordinate uncertainty of ~0.1–0.2 Å at
# typical resolutions. The 8RAT disulfide bonds range from 2.10–2.14 Å
# in the deposited coordinates — all real bonds, but all above a 2.1 Å
# cutoff. A threshold of 2.2 Å catches them all while still being far
# below the nearest non-bonded Cys–Cys distances (> 4 Å).
#
# Printing all distances (not just bonded pairs) makes this transparent:
# students can see the clear gap between bonded and non-bonded pairs.
# -----------------------------------------------------------------------

def find_disulfide_bonds(structure, threshold=2.2):
    '''
    Search a protein structure for disulfide bonds between Cysteine residues.

    Disulfide bonds form when two Cys SG (sulfur gamma) atoms are within
    approximately 2.03–2.15 Å in crystal structures. This function prints
    the distance for every unique Cys pair so you can verify the threshold
    is correct, then returns only the pairs that fall within the threshold.

    Parameters:
    -----------
    structure : Bio.PDB.Structure.Structure
        The parsed BioPython structure object
    threshold : float
        Maximum SG–SG distance (Å) to call a disulfide bond (default 2.2 Å).
        This is slightly above the theoretical S–S bond length of ~2.04 Å
        to account for coordinate uncertainty in crystal structures.

    Returns:
    --------
    list of dicts
        Each dict has keys: Chain1, Residue1, Chain2, Residue2, Distance_A
    '''
    # Step 1: Collect all Cys SG atoms across every chain
    cysteine_sg_list = []

    for model in structure:
        for chain in model:
            for residue in chain:
                if residue.id[0] != ' ':
                    continue                       # skip HETATM records
                if residue.get_resname() != 'CYS':
                    continue                       # skip non-Cys residues
                if 'SG' not in residue:
                    continue                       # skip Cys with no SG atom

                sg_atom = residue['SG']
                cysteine_sg_list.append({
                    'chain':   chain.id,
                    'res_num': residue.id[1],
                    'sg_atom': sg_atom
                })

    print(f'Found {len(cysteine_sg_list)} Cysteine residues with SG atoms:')
    for cys in cysteine_sg_list:
        print(f'  Chain {cys["chain"]}, Cys{cys["res_num"]}')
    print()

    # Step 2: Calculate and print the distance for EVERY unique pair
    # The disulfide-bonded pairs should stand out clearly (~2.1 Å)
    # while non-bonded pairs will be much farther apart (often > 10 Å)
    print('All Cys SG–SG pairwise distances:')
    print(f'  {"Pair":<30} {"Distance (Å)":>12}  {"Disulfide?"}')
    print(f'  {"-"*30} {"-"*12}  {"-"*10}')

    disulfide_bonds = []

    for i in range(len(cysteine_sg_list)):
        for j in range(i + 1, len(cysteine_sg_list)):
            cys_i = cysteine_sg_list[i]
            cys_j = cysteine_sg_list[j]

            sg_distance = calculate_distance(cys_i['sg_atom'], cys_j['sg_atom'])

            pair_label = (f"Cys{cys_i['res_num']}({cys_i['chain']}) — "
                          f"Cys{cys_j['res_num']}({cys_j['chain']})")
            is_disulfide = 'YES ✓' if sg_distance <= threshold else ''
            print(f'  {pair_label:<30} {sg_distance:>12.3f}  {is_disulfide}')

            if sg_distance <= threshold:
                disulfide_bonds.append({
                    'Chain1':     cys_i['chain'],
                    'Residue1':   cys_i['res_num'],
                    'Chain2':     cys_j['chain'],
                    'Residue2':   cys_j['res_num'],
                    'Distance_A': round(sg_distance, 3)
                })

    print()
    return disulfide_bonds


print('find_disulfide_bonds() function defined.')

In [ ]:
# -----------------------------------------------------------------------
# Apply find_disulfide_bonds() to RNase A (8RAT)
# We expect to find all four disulfide bonds: 26-84, 40-95, 58-110, 65-72
# -----------------------------------------------------------------------

print('Searching for disulfide bonds in RNase A (8RAT)...')
print('Threshold: SG–SG distance ≤ 2.2 Å')
print()

disulfide_bonds = find_disulfide_bonds(structure, threshold=2.2)

print(f'Summary: {len(disulfide_bonds)} disulfide bond(s) found (SG–SG ≤ 2.2 Å)')
print()

if len(disulfide_bonds) > 0:
    disulfide_df = pd.DataFrame(disulfide_bonds)
    print(disulfide_df.to_string(index=False))
    print()
    print('Expected bonds: Cys26–Cys84, Cys40–Cys95, Cys58–Cys110, Cys65–Cys72')
    print()
    print('The four bonded pairs all fall between 2.10–2.15 Å — far shorter than')
    print('any non-bonded pair. This gap makes the 2.2 Å threshold easy to justify:')
    print('the nearest non-bonded pair is >4 Å away.')
    print()
    print('All 8 Cysteine residues in RNase A are involved in disulfide bonds.')
    print('This is unusual: many proteins have free (unbonded) cysteines as well.')

In [ ]:
# -----------------------------------------------------------------------
# Visualize the disulfide bonds in the RNase A 3D structure
#
# The cartoon is shown at low opacity colored by secondary structure.
# The 8 Cys residues are highlighted as ball-and-stick with element
# coloring: sulfur atoms appear yellow, carbon gray, nitrogen blue.
# You should be able to see the four S–S bonds connecting paired Cys.
# -----------------------------------------------------------------------

view = nv.show_structure_file(
    pdb_filename,
    representations=[{
        'type': 'cartoon',
        'params': {'colorScheme': 'sstruc', 'opacity': 0.4}
    }]
)

# Highlight all Cysteine residues as ball-and-stick
# '[CYS]' selects all residues named CYS in NGL selection syntax
# colorScheme='element' gives: C=gray, N=blue, O=red, S=yellow
view.add_ball_and_stick('[CYS]', colorScheme='element')

view.background = 'white'
view

### 💭 Think About It

1. **Your function found exactly four disulfide bonds**, matching the known structure of RNase A. Notice that the bonded pairs range from 2.10–2.14 Å — slightly above the theoretical S–S bond length of 2.04 Å. This is normal: crystal structure coordinates have uncertainty of ~0.1–0.2 Å at typical X-ray resolutions. What if the threshold were too tight (e.g., 2.0 Å) or too loose (e.g., 4.0 Å)?

2. **Rotate the 3D structure and find the four pairs of yellow sulfur atoms.** Are the bonded Cys pairs close in sequence (small difference in residue number) or far apart? Which bonds cross the largest distances along the sequence? How does a disulfide bond that connects distant parts of the sequence constrain the overall fold?

3. **Anfinsen's experiment:** When RNase A is denatured with urea and beta-mercaptoethanol (which breaks all disulfide bonds), there are 105 possible ways to pair 8 cysteines into 4 bonds. Yet when the denaturant is removed, the correct four bonds reform spontaneously. What does this tell you about the relationship between amino acid sequence, 3D structure, and disulfide bonding?

4. **In the Try It Yourself section, you will analyze insulin**, which also has disulfide bonds. Insulin has only 6 cysteines forming 3 disulfide bonds — and two of the three bonds connect the A chain to the B chain. How do you predict this will affect the stability of the insulin structure compared to RNase A, where all bonds are within one chain?

---
## Section 3: Identifying Ionic Interactions

### What is an Ionic Interaction?

An **ionic interaction** (also widely known as a **salt bridge** in the structural biology literature) forms between two residue side chains that carry **opposite charges** at physiological pH:

**Positively charged residues** (at pH ~7):
- **Arginine (Arg, R):** the guanidinium group carries a +1 charge; key atoms: NE, NH1, NH2
- **Lysine (Lys, K):** the terminal amino group (–NH₃⁺) carries a +1 charge; key atom: NZ

**Negatively charged residues** (at pH ~7):
- **Aspartate (Asp, D):** the carboxylate group (–COO⁻) carries a −1 charge; key atoms: OD1, OD2
- **Glutamate (Glu, E):** the carboxylate group (–COO⁻) carries a −1 charge; key atoms: OE1, OE2

An ionic interaction is typically defined as a charged nitrogen or oxygen from one group coming within **4.0 Å** of a charged atom from an oppositely charged group.

Ionic interactions contribute to:
- **Protein stability** — they stabilize the folded state relative to the unfolded state
- **Enzyme function** — charged residues in active sites often participate in catalysis (e.g., Lys41 in RNase A stabilizes the transition state)
- **pH sensitivity** — if a residue's charge changes with pH, ionic interactions are pH-dependent

> **Terminology note:** You will see "ionic interaction," "salt bridge," and "electrostatic interaction" used interchangeably in the literature. They all refer to the same phenomenon. "Salt bridge" is the most common informal term in structural biology papers.

In [ ]:
# -----------------------------------------------------------------------
# Define which atoms carry charge in each residue type
# -----------------------------------------------------------------------

POSITIVE_ATOMS = {
    'ARG': ['NE', 'NH1', 'NH2'],   # Arginine: guanidinium nitrogens
    'LYS': ['NZ'],                  # Lysine: terminal amino nitrogen
}

NEGATIVE_ATOMS = {
    'ASP': ['OD1', 'OD2'],   # Aspartate: carboxylate oxygens
    'GLU': ['OE1', 'OE2'],   # Glutamate: carboxylate oxygens
}


def find_ionic_interactions(structure, threshold=4.0):
    '''
    Identify ionic interactions between oppositely charged residue side chains.

    An ionic interaction (also called a salt bridge) is defined as a charged
    atom from a positively charged residue within `threshold` Angstroms of a
    charged atom from a negatively charged residue.

    Parameters:
    -----------
    structure : Bio.PDB.Structure.Structure
        The parsed BioPython structure object
    threshold : float
        Maximum distance (Å) between charged atoms to call an ionic
        interaction (default 4.0 Å, the standard geometric criterion)

    Returns:
    --------
    list of dicts, each describing one interaction pair with keys:
        PosChain, PosResidue, PosName, PosAtom,
        NegChain, NegResidue, NegName, NegAtom, Distance_A
    '''
    positive_atom_list = []
    negative_atom_list = []

    for model in structure:
        for chain in model:
            for residue in chain:
                if residue.id[0] != ' ':
                    continue
                resname = residue.get_resname()

                if resname in POSITIVE_ATOMS:
                    for atom_name in POSITIVE_ATOMS[resname]:
                        if atom_name in residue:
                            positive_atom_list.append({
                                'chain':   chain.id,
                                'res_num': residue.id[1],
                                'resname': resname,
                                'atom':    residue[atom_name]
                            })

                if resname in NEGATIVE_ATOMS:
                    for atom_name in NEGATIVE_ATOMS[resname]:
                        if atom_name in residue:
                            negative_atom_list.append({
                                'chain':   chain.id,
                                'res_num': residue.id[1],
                                'resname': resname,
                                'atom':    residue[atom_name]
                            })

    print(f'Positive residue atoms found: {len(positive_atom_list)} (from Arg and Lys)')
    print(f'Negative residue atoms found: {len(negative_atom_list)} (from Asp and Glu)')
    print(f'Checking {len(positive_atom_list) * len(negative_atom_list):,} atom pairs...')
    print()

    ionic_interactions = []

    for pos in positive_atom_list:
        for neg in negative_atom_list:
            if pos['chain'] == neg['chain'] and pos['res_num'] == neg['res_num']:
                continue   # skip same residue

            dist = calculate_distance(pos['atom'], neg['atom'])

            if dist <= threshold:
                ionic_interactions.append({
                    'PosChain':   pos['chain'],
                    'PosResidue': pos['res_num'],
                    'PosName':    pos['resname'],
                    'PosAtom':    pos['atom'].get_name(),
                    'NegChain':   neg['chain'],
                    'NegResidue': neg['res_num'],
                    'NegName':    neg['resname'],
                    'NegAtom':    neg['atom'].get_name(),
                    'Distance_A': round(dist, 3)
                })

    return ionic_interactions


print('find_ionic_interactions() function defined.')

In [ ]:
# -----------------------------------------------------------------------
# Apply find_ionic_interactions() to RNase A (8RAT)
# -----------------------------------------------------------------------

print('Searching for ionic interactions in RNase A (8RAT)...')
print('Threshold: charged-atom distance ≤ 4.0 Å')
print()

all_ionic_interactions = find_ionic_interactions(structure, threshold=4.0)

print(f'Total ionic interaction contacts found: {len(all_ionic_interactions)}')
print()

# Convert to DataFrame and sort by distance
ii_df = pd.DataFrame(all_ionic_interactions)
ii_df = ii_df.sort_values('Distance_A').reset_index(drop=True)

# Deduplicate: keep shortest distance for each unique residue pair
ii_df['pair_key'] = ii_df.apply(
    lambda row: tuple(sorted([
        f"{row['PosChain']}{row['PosResidue']}",
        f"{row['NegChain']}{row['NegResidue']}"
    ])),
    axis=1
)
unique_ii_df = ii_df.drop_duplicates(subset='pair_key').drop(columns='pair_key')
unique_ii_df = unique_ii_df.reset_index(drop=True)

print(f'Unique ionic interaction residue pairs: {len(unique_ii_df)}')
print()
print('Shortest (strongest) 15 ionic interactions:')
print('=' * 75)
print(unique_ii_df.head(15).to_string(index=False))
print()

# Count by residue type pair
print('Ionic interactions by residue type pair:')
print('-' * 40)
type_pair_counts = ii_df.groupby(['PosName', 'NegName']).size().reset_index(name='Count')
type_pair_counts = type_pair_counts.sort_values('Count', ascending=False)
print(type_pair_counts.to_string(index=False))

In [ ]:
# -----------------------------------------------------------------------
# Visualize ionic interaction partners in the RNase A 3D structure
#
# The cartoon is shown at low opacity so you can see the residue side chains.
# Positively charged residues (Arg, Lys) are shown in blue.
# Negatively charged residues (Asp, Glu) are shown in red.
# When two residues of opposite color appear close together, that is an
# ionic interaction.
# -----------------------------------------------------------------------

view = nv.show_structure_file(
    pdb_filename,
    representations=[{
        'type': 'cartoon',
        'params': {'colorScheme': 'sstruc', 'opacity': 0.4}
    }]
)

# Show Arg and Lys side chains in blue (positively charged)
# NGL selection: '[ARG] or [LYS]' selects residues named ARG or LYS
view.add_licorice('[ARG] or [LYS]', color='blue')

# Show Asp and Glu side chains in red (negatively charged)
view.add_licorice('[ASP] or [GLU]', color='red')

view.background = 'white'

view

### 💭 Think About It

1. **How many unique ionic interaction pairs did you find in RNase A?** At roughly 1 ionic interaction per 10–15 residues, how does your count compare to that estimate for a 124-residue protein?

2. **Look at the residue pair type chart.** Are Arg–Asp or Arg–Glu interactions more common than Lys–Asp or Lys–Glu? Arginine has three charged nitrogen atoms (NE, NH1, NH2), while Lysine has only one (NZ). How might this geometry difference affect the number of contacts detected?

3. **Ionic interactions are sometimes described as stabilizing and sometimes as destabilizing.** An ionic interaction on the protein surface competes with water molecules for the charged groups. An ionic interaction buried in the hydrophobic core is stronger (less competition from water). From the 3D view, can you identify any ionic interactions that appear to be on the surface versus buried?

4. **Lys41 is critical for RNase A catalysis** — it stabilizes the negatively charged transition state of the RNA cleavage reaction. Check the results table: does Lys41 appear as a partner in any of the ionic interactions you found? Why might a catalytic lysine form an ionic interaction with a nearby acidic residue at resting pH, but not during catalysis?

---
## Section 4: Domain Organization

### What is a Protein Domain?

A **protein domain** is an independently folding, structurally compact unit of a polypeptide chain. Domains are the fundamental modular units of protein evolution:

- A single polypeptide may fold into **one or several domains**, each with its own hydrophobic core
- Domains often fold and unfold independently in denaturation experiments
- The same domain fold can appear in many different proteins ("domain shuffling")
- Domains typically correspond to discrete functions: a binding domain, a catalytic domain, a regulatory domain, etc.

### Databases: CATH and SCOP

Two major databases classify protein domains by structural similarity:

| Database | Full name | Classification levels |
|---|---|---|
| **CATH** | Class, Architecture, Topology, Homology | 4 hierarchical levels |
| **SCOP** | Structural Classification Of Proteins | Class, Fold, Superfamily, Family |

### A Multi-Domain Example: Human c-Src Tyrosine Kinase (2SRC)

**RNase A** (8RAT), studied throughout this notebook, is a **single-domain protein** — the entire 124-residue chain folds into one compact unit with a single entry in CATH.

By contrast, many signaling proteins contain multiple domains that act as an integrated molecular machine. A textbook example is **human c-Src tyrosine kinase** (PDB: **2SRC**) — the founding member of the Src-family kinases and the first proto-oncogene ever identified. Src adds a phosphate group to **tyrosine** residues on target proteins, triggering cascades that control cell growth, adhesion, and survival.

The crystal structure of the **inactive (autoinhibited)** form of c-Src reveals three structural domains arranged along a single polypeptide chain:

| Domain | Residues | Fold | Function |
|--------|----------|------|----------|
| **SH3** (Src Homology 3) | ~83–143 | β-barrel | Recognizes proline-rich **PXXP** motifs on partner proteins; holds the linker in an inhibited conformation |
| **SH2** (Src Homology 2) | ~154–248 | α/β sandwich | Recognizes **phosphotyrosine** sequences; in the inactive state, binds c-Src's own C-terminal pTyr527, locking the enzyme off |
| **Kinase (SH1)** | ~267–533 | Bilobed α/β | The catalytic domain; transfers the γ-phosphate of ATP to a substrate tyrosine |

### Autoinhibition: Domains Talking to Each Other

The 2SRC structure captures a beautiful example of **intramolecular regulation**:

1. A regulatory kinase (Csk) phosphorylates **Tyr527** in the c-Src C-terminal tail
2. The **SH2 domain** binds its own pTyr527, folding the tail back against the kinase domain
3. Simultaneously, the **SH3 domain** grips the SH2–kinase linker, further clamping the structure closed
4. Both interactions hold the kinase domain in an **inactive conformation**

To activate Src, either grip must be broken — by dephosphorylating Tyr527 (a phosphatase), or by a higher-affinity external SH2 or SH3 ligand competing for those binding surfaces. Mutations that disrupt the autoinhibitory contacts produce **constitutively active** Src, which drives uncontrolled cell proliferation. This is why the viral oncogene v-Src (which lacks the C-terminal tail entirely) is so potent.

> **Cancer connection:** Src-family kinase overactivation is found in many human cancers. Imatinib (Gleevec) and dasatinib are FDA-approved kinase inhibitors that target the ATP-binding cleft of the kinase domain — the same cleft you will see in the visualization below.

In [ ]:
# -----------------------------------------------------------------------
# Query the CATH API for domain information about c-Src (2SRC)
#
# CATH classifies each structural domain independently.
# 2SRC has three main domains (SH3, SH2, Kinase), each with a completely
# different fold — a striking contrast to single-domain RNase A (8RAT).
# -----------------------------------------------------------------------

pdb_id = '2src'   # CATH uses lowercase PDB IDs
cath_url = f'https://www.cathdb.info/version/v4_3_0/api/rest/pdb/{pdb_id}'

print(f'Querying CATH API for {pdb_id.upper()}...')
print(f'URL: {cath_url}')
print()

try:
    response = requests.get(cath_url, timeout=15)

    if response.status_code == 200:
        cath_data = response.json()
        domain_list = cath_data.get('data', [])

        print(f'CATH returned {len(domain_list)} domain record(s) for {pdb_id.upper()}:')
        print()
        print(f'  {"Domain ID":<15} {"CATH Code":<20} {"Class":<30}')
        print(f'  {"-"*15} {"-"*20} {"-"*30}')

        for domain in domain_list:
            domain_id  = domain.get('domain', 'N/A')
            cath_code  = domain.get('cathCode', 'N/A')
            class_name = domain.get('className', '')
            print(f'  {domain_id:<15} {cath_code:<20} {class_name:<30}')

    else:
        raise ConnectionError(f'HTTP status {response.status_code}')

except Exception as api_error:
    print(f'CATH API not reachable: {api_error}')
    print()
    print('Using known CATH domain classification for 2SRC (CATH v4.3):')
    print()
    print(f'  {"Domain ID":<15} {"CATH Code":<20} {"Domain / Fold"}')
    print(f'  {"-"*15} {"-"*20} {"-"*48}')
    print(f'  {"2srca01":<15} {"2.30.30.40":<20} SH3 domain — SH3-type beta barrel')
    print(f'  {"2srca02":<15} {"3.30.505.10":<20} SH2 domain — alpha/beta phosphotyrosine-binding')
    print(f'  {"2srca03":<15} {"1.10.510.10":<20} Kinase domain — protein kinase bilobed alpha/beta')

### What the CATH Codes Tell Us

The table above gives CATH IDs and codes — but what do those codes actually mean biologically? In a research setting you would look each code up on [cathdb.info](https://www.cathdb.info) or in the original paper describing the structure. Here is the interpretation for c-Src:

**SH3 domain (CATH 2.30.30.40) — Mainly-beta, SH3-type barrel**
Six antiparallel beta strands form a compact barrel with a surface groove lined with hydrophobic residues. This groove binds **PXXP** proline-rich motifs on partner proteins. SH3 domains appear in over 400 human proteins — one of the most widely recycled domain folds in eukaryotic cell signaling.

**SH2 domain (CATH 3.30.505.10) — Alpha/Beta sandwich**
A central beta sheet flanked by two alpha helices. A positively charged pocket grips a **phosphotyrosine** (pTyr) residue and reads the two or three flanking residues to achieve binding specificity. In the inactive 2SRC structure, this pocket is occupied by c-Src's own C-terminal pTyr527 — the enzyme holding itself off.

**Kinase domain (CATH 1.10.510.10) — Protein kinase bilobed fold**
An N-terminal lobe (mostly beta sheets) binds ATP; a C-terminal lobe (mostly alpha helices) binds the protein substrate. Catalysis — transfer of the γ-phosphate from ATP to a substrate tyrosine — occurs in the cleft between the two lobes. In the 2SRC inactive structure, the activation loop is unphosphorylated and blocks the substrate-binding site.

> **Key contrast:** RNase A (8RAT) has **one** CATH domain and one function (RNA hydrolysis). c-Src (2SRC) has **three** CATH domains with three distinct folds — recognition (SH3), regulation (SH2), and catalysis (kinase) — all encoded in a single polypeptide chain.

In [ ]:
# -----------------------------------------------------------------------
# Download c-Src (2SRC) and visualize each domain in a different color
#
# The key technique here is NGL's residue-range selection syntax:
#   '83-143:A'  = residues 83 through 143 in chain A
#
# We load the structure with a nearly invisible gray base cartoon so
# the short linker segments between domains are still shown as a thin
# thread. Then we add three brightly colored cartoon representations,
# one per domain, layered on top.
#
# Domain boundaries (from CATH and the original Xu et al. 1997 paper):
#   SH3 domain  : residues  83–143  → blue   (#4472C4)
#   SH2 domain  : residues 154–248  → green  (#70AD47)
#   Kinase domain: residues 267–516  → orange (#FF6600)
#   (Linker regions 144–153 and 249–266 appear as faint gray)
# -----------------------------------------------------------------------

from IPython.display import display

pdbl_src = PDBList()

try:
    src_downloaded_path = pdbl_src.retrieve_pdb_file(
        '2SRC',
        file_format='pdb',
        pdir='.'
    )
except Exception as download_error:
    raise RuntimeError(
        'Could not download 2SRC from RCSB PDB.\n'
        'Check your internet connection and try again.\n'
        f'Original error: {download_error}'
    )

# Search for the file if not at the expected path
if not os.path.exists(src_downloaded_path):
    for root, dirs, files in os.walk('.'):
        for fname in files:
            if '2src' in fname.lower():
                src_downloaded_path = os.path.join(root, fname)
                break

src_filename = src_downloaded_path

# Parse and report structure contents
src_parser = PDBParser(QUIET=True)
src_structure = src_parser.get_structure('2SRC', src_filename)
src_model = src_structure[0]

src_chains = [chain.id for chain in src_model.get_chains()]
print(f'Chains in 2SRC: {src_chains}')

# Report the residue range actually present in the structure
for chain in src_model.get_chains():
    aa_residues = [r for r in chain.get_residues() if r.id[0] == ' ']
    if aa_residues:
        first_res = aa_residues[0].id[1]
        last_res  = aa_residues[-1].id[1]
        print(f'Chain {chain.id}: residues {first_res}–{last_res} ({len(aa_residues)} amino acids)')

print()

# -----------------------------------------------------------------------
# Visualization: cartoon colored by domain using residue-range selections
#
# Step 1: Load the full structure with a nearly transparent gray base.
#         This ensures linker regions between domains remain visible
#         as a thin thread connecting the three colored domains.
# Step 2: Add a bright cartoon for each domain using 'start-end:chain'
#         selection syntax. Each add_cartoon() call layers a new
#         representation on top of the existing ones.
# -----------------------------------------------------------------------

# Base representation: full chain in light gray at very low opacity
# This shows the linker segments that connect the three domains
view = nv.show_structure_file(
    src_filename,
    representations=[{
        'type': 'cartoon',
        'params': {'color': '#aaaaaa', 'opacity': 0.2}
    }]
)

# SH3 domain — BLUE
# Six-stranded antiparallel beta barrel; recognizes PXXP proline-rich motifs
view.add_cartoon('83-143:A', color='#4472C4')

# SH2 domain — GREEN
# Alpha/beta phosphotyrosine-binding domain; in 2SRC holds own pTyr527
view.add_cartoon('154-248:A', color='#70AD47')

# Kinase (SH1) domain — ORANGE
# Bilobed ATP/substrate-binding catalytic domain
view.add_cartoon('267-516:A', color='#FF6600')

view.background = 'white'

print('Color key:')
print('  Blue   (#4472C4) — SH3 domain   (residues  83–143):  beta barrel')
print('  Green  (#70AD47) — SH2 domain   (residues 154–248):  alpha/beta sandwich')
print('  Orange (#FF6600) — Kinase domain (residues 267–516):  bilobed alpha/beta')
print('  Gray   (faint)   — Linker regions connecting the domains')
print()
print('Rotate the structure to see all three domains. Notice how the SH2 and')
print('SH3 domains (green and blue) pack against the back of the kinase domain')
print('(orange) — this is the autoinhibited conformation captured in 2SRC.')
print()

view

### 💭 Think About It

1. **The CATH query returned three domain entries for 2SRC but only one for RNase A.** Both are single-chain proteins. What structural feature causes CATH to classify a chain as having multiple domains? Look at the 3D view — can you identify where one domain ends and the next begins? What do the short gray linker segments between domains look like compared to the folded domain cores?

2. **SH2 and SH3 domains appear in hundreds of human signaling proteins** — not just Src-family kinases, but also PI3K, PLCγ, Grb2, and many others. This is called **domain shuffling**: the same domain fold gets copied, mutated, and inserted into different protein contexts over evolutionary time. What advantage would domain shuffling provide? If two unrelated proteins both contain an SH2 domain, would you expect them to bind the same phosphotyrosine-containing sequences?

3. **The 2SRC structure shows the autoinhibited (inactive) form of c-Src.** In this conformation, the SH2 domain (green) binds the enzyme's own C-terminal pTyr527, and the SH3 domain (blue) grips the SH2–kinase linker. From the 3D view, do the blue and green domains appear to contact the orange kinase domain? How is this type of regulation — a domain binding to its own protein — fundamentally different from a small-molecule inhibitor binding to an enzyme?

4. **The oncogenic v-Src protein lacks the C-terminal tail containing Tyr527.** Predict what happens to the autoinhibition mechanism in v-Src. Based on the domain organization you see in 2SRC, which binding interaction is lost? Why would this make v-Src constitutively active? Why would constitutive kinase activity drive cancer?

5. **Kinase inhibitors like imatinib (Gleevec) and dasatinib bind in the cleft between the N-lobe and C-lobe of the kinase domain (orange).** They compete with ATP for binding. Based on what you know about how enzymes work, why would blocking ATP binding shut off kinase activity? Would you predict that a mutation in the SH2 or SH3 domain would affect the drug's ability to bind?

---
## Summary

In this notebook, you:

- **Accessed** the 3D coordinates (x, y, z) of individual atoms from the 8RAT crystal structure using BioPython's SMCRA hierarchy
- **Derived** the Euclidean distance formula step-by-step and implemented it as the reusable `calculate_distance()` function
- **Measured** biologically meaningful distances — the N-to-C span of RNase A and the separation between the three active-site residues (His12, Lys41, His119)
- **Learned nglview** — how to set representations at initialization, apply color schemes (sstruc, chainindex, element), and build atom selections to highlight specific residues; used residue-range selections (`'83-143:A'`) to color individual domains
- **Searched** for disulfide bonds in RNase A using `find_disulfide_bonds()`, confirmed all four bonds (Cys26–84, Cys40–95, Cys58–110, Cys65–72), and visualized them with element coloring to show the yellow sulfur atoms
- **Identified** ionic interactions in RNase A using `find_ionic_interactions()`, examined the distribution of distances and residue pair types (Arg/Lys ↔ Asp/Glu), and visualized charged side chains in 3D
- **Queried** the CATH database for domain classifications and contrasted two proteins: RNase A (8RAT) — a single-domain enzyme with one CATH entry — versus human c-Src tyrosine kinase (2SRC) — three structurally distinct domains (SH3 beta barrel, SH2 alpha/beta sandwich, kinase bilobed fold) arranged in an autoinhibited signaling machine

### Looking Ahead

In **Notebook 4**, we move to **quaternary structure** — the assembly of multiple polypeptide chains into a functional complex. We will use hemoglobin (1HHO) to examine the interface between the alpha and beta subunits, calculate buried surface area, and explore how cooperative oxygen binding emerges from the quaternary structure.

---
## Try It Yourself: Disulfide Bonds in Insulin

Insulin is the pancreatic hormone that regulates blood glucose. It is synthesized inside the cell as a single chain (proinsulin), then processed and **secreted** into the bloodstream — an oxidizing environment. As a result, insulin has **three disulfide bonds** that are essential for its structure and stability:

| Bond | Residues | Location |
|------|----------|----------|
| 1 | Cys6(A)–Cys11(A) | Intra-chain within the A chain |
| 2 | Cys7(A)–Cys7(B) | Inter-chain between A and B chains |
| 3 | Cys20(A)–Cys19(B) | Inter-chain between A and B chains |

**PDB ID:** `3I3Z` (human insulin, monomeric form, 1.85 Å resolution)

The asymmetric unit of 3I3Z contains two chains: Chain A (21 residues) and Chain B (30 residues).

### Your Tasks

1. **Download and parse 3I3Z** using `PDBParser` and `PDBList`, just as you did for 8RAT. How many amino acids are in Chain A and Chain B?

2. **Run `find_disulfide_bonds()`** on the insulin structure with a threshold of 2.2 Å. How many disulfide bonds does it find? Do the chain and residue numbers match the table above?

3. **Print the SG–SG distances** for the three disulfide-bonded pairs. How do these distances compare to the SG–SG distances you measured in RNase A?

4. **Visualize the insulin structure in nglview.** Use `colorScheme='chainindex'` for the cartoon, and highlight the Cys residues with `add_ball_and_stick('[CYS]', color='yellow')`. Insulin is much smaller than RNase A — can you see the two disulfide bonds that cross between the A and B chains?

5. **Why are disulfide bonds so important for insulin?** Insulin is stored in the pancreas and circulates in the blood for minutes to hours before being degraded. If you broke the disulfide bonds experimentally, would you expect insulin to refold correctly when the bonds were re-allowed to form? (Hint: think about Anfinsen's experiment from Section 2 above.)

> **Hint:** You can re-use `calculate_distance()` and `find_disulfide_bonds()` exactly as written above — they work on any BioPython structure object, not just RNase A.

In [ ]:
# -----------------------------------------------------------------------
# Starter code for the insulin analysis
# Fill in each step below.
# -----------------------------------------------------------------------

# Step 1: Download and parse insulin (3I3Z)
# insulin_pdbl = PDBList()
# insulin_path = insulin_pdbl.retrieve_pdb_file('3I3Z', file_format='pdb', pdir='.')
# if not os.path.exists(insulin_path):
#     for root, dirs, files in os.walk('.'):
#         for fname in files:
#             if '3i3z' in fname.lower():
#                 insulin_path = os.path.join(root, fname)
#                 break
# insulin_structure = parser.get_structure('3I3Z', insulin_path)

# Step 2: Explore the chains and residue counts
# insulin_model = insulin_structure[0]
# for chain in insulin_model.get_chains():
#     aa_residues = [r for r in chain.get_residues() if r.id[0] == ' ']
#     print(f'Chain {chain.id}: {len(aa_residues)} residues')

# Step 3: Find disulfide bonds (use the same threshold as RNase A)
# insulin_disulfide_bonds = find_disulfide_bonds(insulin_structure, threshold=2.2)
# print(f'\nDisulfide bonds found: {len(insulin_disulfide_bonds)}')
# for bond in insulin_disulfide_bonds:
#     print(bond)

# Step 4: Visualize
# insulin_view = nv.show_structure_file(
#     insulin_path,
#     representations=[{'type': 'cartoon', 'params': {'colorScheme': 'chainindex'}}]
# )
# insulin_view.add_ball_and_stick('[CYS]', color='yellow')
# insulin_view.background = 'white'
# insulin_view

# Your code here:
